## I. Tiền Xử Lý Dữ Liệu Spam SMS
Pipeline gồm 8 bước: EDA → Xoá duplicate → Encode label → Làm sạch văn bản → Tokenize + Stopwords → Train/Test split → Vector hoá (TF-IDF) → Balance Dataset (SMOTE)

### Cài đặt thư viện

In [1]:
# Chạy cell này một lần để cài đặt
!pip install pandas scikit-learn nltk matplotlib seaborn imbalanced-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\hary0\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Import thư viện

In [2]:
import pandas as pd          # xử lý dữ liệu dạng bảng
import numpy as np           # xử lý mảng, ma trận
import re                    # xử lý text (regex)
import pickle                # lưu/truy xuất file nhị phân
import nltk                  # xử lý ngôn ngữ tự nhiên
from nltk.corpus import stopwords         # loại bỏ stopwords (từ dừng)
from sklearn.model_selection import train_test_split  # chia tập train/test
from sklearn.feature_extraction.text import TfidfVectorizer  # trích xuất đặc trưng TF-IDF
from scipy.sparse import save_npz, hstack as sp_hstack  # lưu ma trận thưa
from imblearn.over_sampling import SMOTE  # balance dataset
import matplotlib.pyplot as plt           # visualize
import seaborn as sns                     # visualize
from collections import Counter           # đếm từ

nltk.download('stopwords')

print('Import xong!')


Import xong!


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hary0\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Bước 0: Load dữ liệu gốc

In [ ]:
# Load CSV
df = pd.read_csv('../data/spam.csv', encoding='latin-1')

# Giữ lại 2 cột label và text, đổi tên text thành message
df = df[['label', 'text']].rename(columns={'text': 'message'})

print(f'Shape: {df.shape}')
df.head(3)


KeyError: "['message'] not in index"

---
### Bước 1: EDA (Exploratory Data Analysis) — Khám phá dữ liệu
Khảo sát phân phối nhãn, độ dài tin nhắn, từ phổ biến và missing values trước khi xử lý.

In [ ]:
# --- Phân phối nhãn (raw) ---
print('Phân phối nhãn (giá trị gốc):')
print(df['label'].value_counts(dropna=False))
print()

# --- Độ dài tin nhắn ---
df['message_length'] = df['message'].astype(str).str.len()

# --- Biểu đồ phân phối nhãn ---
plt.figure(figsize=(6, 4))
df['label'].value_counts().plot(kind='bar')
plt.title('Phân phối nhãn (giá trị gốc)')
plt.xlabel('Nhãn')
plt.ylabel('Số lượng')
plt.tight_layout()
plt.show()

# --- Biểu đồ độ dài tin nhắn ---
plt.figure(figsize=(8, 5))
for lbl in df['label'].unique():
    sns.histplot(df[df['label'] == lbl]['message_length'], label=str(lbl), alpha=0.7, bins=50)
plt.title('Phân phối độ dài tin nhắn')
plt.xlabel('Độ dài (ký tự)')
plt.legend()
plt.tight_layout()
plt.show()

# --- Missing values ---
print('\nMissing values:')
print(df.isnull().sum())

# --- Thống kê cơ bản ---
print(f'\nSố mẫu         : {len(df)}')
print(f'Độ dài trung bình: {df["message_length"].mean():.1f} ký tự')
print(f'Độ dài trung vị  : {df["message_length"].median():.1f} ký tự')


---
### Bước 2: Delete Duplicate
Loại bỏ các dòng trùng lặp để tránh **data leakage** khi chia tập train/test.

In [ ]:
before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after = len(df)
print(f'Trước : {before} dòng')
print(f'Sau   : {after} dòng')
print(f'Đã xoá: {before - after} dòng trùng')


---
### Bước 3: Encode Label
Chuyển nhãn dạng text sang số: `ham → 0`, `spam → 1`.

In [ ]:
if df['label'].dtype == object:
    df['label'] = df['label'].astype(str).str.strip().map({'ham': 0, 'spam': 1})

if df['label'].isnull().any():
    print('Cảnh báo: vẫn còn label NaN sau map. Các giá trị lạ:')
    print(df.loc[df['label'].isnull(), 'label'].head())

print('Phân phối nhãn sau encode:')
print(df['label'].value_counts(dropna=False))
print()
print(f'Null values: {df["label"].isnull().sum()}')
print(f'Tỷ lệ spam : {df["label"].mean():.2%}')


---
### Bước 4: Làm sạch văn bản
Chuyển lowercase, loại URL, email, số điện thoại, ký tự đặc biệt.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    # Loại bỏ URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Loại bỏ email
    text = re.sub(r'\S+@\S+', '', text)
    # Loại bỏ số điện thoại
    text = re.sub(r'\b\d{10,}\b', '', text)
    # Loại bỏ ký tự đặc biệt, giữ lại chữ và số
    text = re.sub(r'[^\w\s]', '', text)
    # Loại bỏ khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_msg'] = df['message'].apply(clean_text)

print('Đã clean text xong!')
print(f'Ví dụ: {df["clean_msg"].iloc[0][:100]}...')


---
### Bước 5: Tokenize + Loại Stopwords
Tách từ và loại bỏ các từ phổ biến không mang nhiều ý nghĩa (the, is, a, to...).

In [ ]:
stop_words = set(stopwords.words('english'))

def tokenize_and_remove_stopwords(text):
    tokens = str(text).split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

df['processed'] = df['clean_msg'].apply(tokenize_and_remove_stopwords)

# Thống kê
avg_before = df['clean_msg'].str.split().apply(len).mean()
avg_after  = df['processed'].str.split().apply(len).mean()
print(f'Độ dài TB trước: {avg_before:.1f} từ/tin nhắn')
print(f'Độ dài TB sau  : {avg_after:.1f} từ/tin nhắn')
print()

# Xem ví dụ
print('Ví dụ trước/sau tokenize:\n')
for i in [0, 2, 10]:
    print(f'  TRƯỚC: {df["clean_msg"].iloc[i][:90]}')
    print(f'  SAU  : {df["processed"].iloc[i][:90]}')
    print()


---
### Feature Engineering
Tạo các đặc trưng số để bổ sung cho TF-IDF, giúp mô hình phân biệt ham/spam tốt hơn.

In [ ]:
# 1. Số từ trong tin nhắn
df['word_count'] = df['clean_msg'].apply(lambda x: len(str(x).split()))

# 2. Số từ viết hoa
df['uppercase_words'] = df['message'].apply(
    lambda x: len([w for w in str(x).split() if w.isupper() and len(w) > 1]))

# 3. Tỷ lệ từ viết hoa
df['uppercase_ratio'] = df['uppercase_words'] / (df['word_count'] + 1)

# 4. Số ký tự đặc biệt
df['special_chars'] = df['message'].apply(lambda x: len(re.findall(r'[^\w\s]', str(x))))

# 5. Số chữ số
df['digit_count'] = df['message'].apply(lambda x: len(re.findall(r'\d', str(x))))

# 6. Số từ dài (>6 ký tự)
df['long_words'] = df['clean_msg'].apply(
    lambda x: len([w for w in str(x).split() if len(w) > 6]))

# 7. Số spam keywords
spam_keywords = ['free', 'win', 'prize', 'urgent', 'call', 'text', 'claim', 'cash', 'money', 'offer']
df['spam_keywords'] = df['clean_msg'].apply(
    lambda x: sum(1 for word in str(x).split() if word.lower() in spam_keywords))

# 8. Tỷ lệ ký tự đặc biệt
df['special_ratio'] = df['special_chars'] / (df['message_length'] + 1)

# 9. Số câu
df['sentence_count'] = df['message'].apply(lambda x: len(re.findall(r'[.!?]', str(x))))

# 10. Độ dài trung bình từ
df['avg_word_length'] = df['clean_msg'].apply(
    lambda x: np.mean([len(w) for w in str(x).split()]) if str(x).split() else 0)

# 11. Số dấu chấm than
df['exclamation_count'] = df['message'].apply(lambda x: str(x).count('!'))

# 12. Số dấu hỏi
df['question_count'] = df['message'].apply(lambda x: str(x).count('?'))

# 13. Dấu câu lặp liên tiếp
def count_consecutive_punct(text):
    return len(re.findall(r'([.!?])\1{1,}', str(text)))
df['consecutive_punct'] = df['message'].apply(count_consecutive_punct)

# 14. Ký hiệu tiền tệ
currency_symbols = ['$', '£', '€', '¥']
df['currency_symbol_count'] = df['message'].apply(
    lambda x: sum(str(x).count(s) for s in currency_symbols))

# 15. Spam tags cụm từ
spam_tags = ['act now', 'guaranteed', 'limited time', 'buy now', 'click here', 'urgent', 'risk free']
df['has_spam_tag'] = df['clean_msg'].apply(
    lambda x: any(tag in str(x).lower() for tag in spam_tags)).astype(int)

NUMERIC_FEATURES = [
    'message_length', 'word_count', 'uppercase_words', 'uppercase_ratio',
    'special_chars', 'digit_count', 'long_words', 'spam_keywords',
    'special_ratio', 'sentence_count', 'avg_word_length',
    'exclamation_count', 'question_count', 'consecutive_punct',
    'currency_symbol_count', 'has_spam_tag'
]

print(f'Đã tạo {len(NUMERIC_FEATURES)} đặc trưng số!')
print('Danh sách:', NUMERIC_FEATURES)


#### Visualize đặc trưng theo nhãn

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()

for i, feature in enumerate(NUMERIC_FEATURES):
    if i < len(axes):
        sns.boxplot(x='label', y=feature, data=df, ax=axes[i])
        axes[i].set_title(f'{feature} by Label')
        axes[i].set_xlabel('Label (0=ham, 1=spam)')

# Ẩn ô thừa
for j in range(len(NUMERIC_FEATURES), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

print('Thống kê trung bình các đặc trưng theo nhãn:')
print(df.groupby('label')[NUMERIC_FEATURES].mean().round(3))


---
### Bước 6: Train / Test Split
Chia 80/20 với `stratify=label` để giữ nguyên tỉ lệ lớp ham/spam trong cả hai tập.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['processed'],
    df['label'],
    test_size=0.2,
    stratify=df['label'],
    random_state=42
)

# Tách numeric features tương ứng
X_train_num = df.loc[X_train.index, NUMERIC_FEATURES].values
X_test_num  = df.loc[X_test.index,  NUMERIC_FEATURES].values

print(f'Train set : {len(X_train)} mẫu')
print(f'Test set  : {len(X_test)} mẫu')
print()
print('Tỉ lệ lớp trong train:')
print(y_train.value_counts(normalize=True).round(3))
print()
print('Tỉ lệ lớp trong test:')
print(y_test.value_counts(normalize=True).round(3))


---
### Bước 7: Vector Hoá (TF-IDF)
Chuyển văn bản thành vector số.

> **Quan trọng:** Chỉ `fit_transform` trên tập **train**, tập **test** chỉ dùng `transform` — tránh data leakage.

In [ ]:
from scipy.sparse import hstack as sp_hstack
import numpy as np

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train_tfidf = vectorizer.fit_transform(X_train)   # fit + transform trên train
X_test_tfidf  = vectorizer.transform(X_test)        # chỉ transform trên test

print(f'Shape TF-IDF train: {X_train_tfidf.shape}')
print(f'Shape TF-IDF test : {X_test_tfidf.shape}')

# Kết hợp TF-IDF với numeric features
from scipy.sparse import csr_matrix
X_train_combined = sp_hstack([X_train_tfidf, csr_matrix(X_train_num)])
X_test_combined  = sp_hstack([X_test_tfidf,  csr_matrix(X_test_num)])

print(f'\nShape sau khi kết hợp TF-IDF + numeric features:')
print(f'  X_train_combined: {X_train_combined.shape}')
print(f'  X_test_combined : {X_test_combined.shape}')


---
### Bước 8: Balance Dataset (SMOTE)
Sử dụng SMOTE (Synthetic Minority Over-sampling Technique) để tạo mẫu giả cho lớp thiểu số (spam), cân bằng dataset trước khi huấn luyện.

> **Lưu ý:** SMOTE chỉ áp dụng trên tập **train** để tránh data leakage. Tập **test** giữ nguyên không cân bằng để đánh giá thực tế mô hình.

In [ ]:
smote = SMOTE(random_state=42)

X_train_balanced, y_train_balanced = smote.fit_resample(X_train_combined, y_train)

print('Trước SMOTE:')
print(f'  X_train shape: {X_train_combined.shape}')
print(f'  Phân phối:\n  {y_train.value_counts().sort_index().to_dict()}')
print()
print('Sau SMOTE:')
print(f'  X_train_balanced shape: {X_train_balanced.shape}')
counts = pd.Series(y_train_balanced).value_counts().sort_index()
print(f'  Ham (0): {counts[0]} mẫu')
print(f'  Spam (1): {counts[1]} mẫu')


---
### Lưu kết quả

In [ ]:
import os
os.makedirs('../data', exist_ok=True)

# Lưu dataframe đã xử lý
df.to_csv('../data/spam_processed.csv', index=False)

# Lưu train/test text (trước SMOTE, giữ nhãn gốc)
train_df = pd.DataFrame({'text': X_train, 'label': y_train})
test_df  = pd.DataFrame({'text': X_test,  'label': y_test})
train_df.to_csv('../data/train.csv', index=False)
test_df.to_csv('../data/test.csv',   index=False)

# Lưu TF-IDF matrix (train sau SMOTE, test không SMOTE)
save_npz('../data/X_train.npz', X_train_balanced)
save_npz('../data/X_test.npz',  X_test_combined)
np.save('../data/y_train.npy',  y_train_balanced)
np.save('../data/y_test.npy',   y_test.values)

# Lưu vectorizer để dùng lại khi predict
with open('../data/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print('Đã lưu tất cả dữ liệu vào ../data/')
print('  spam_processed.csv       — toàn bộ dataframe đã xử lý')
print('  train.csv / test.csv     — tập train/test text (trước SMOTE)')
print('  X_train.npz / X_test.npz — ma trận TF-IDF + numeric (train sau SMOTE)')
print('  y_train.npy / y_test.npy — nhãn')
print('  tfidf_vectorizer.pkl     — vectorizer dùng khi predict')
